# Resume / CV Generator — Interactive UI

A Gradio-powered web interface that generates a polished, professional resume from your details.

**How to use:**
1. Run all cells below
2. A link will appear — click it to open the UI
3. Paste your **OpenAI API key** (or leave blank and pick *Llama 3.2* for free local generation)
4. Fill in your details (demo data is pre-loaded so you can test instantly)
5. Click **Generate Resume** — your polished resume streams in seconds

**Technique used:** Multi-shot prompting — one complete example resume is included in the message chain so the model follows the exact format every time, no lengthy instructions needed.

In [ ]:
# Uncomment and run this line if gradio is not installed
# !pip install gradio

In [1]:
import gradio as gr
from openai import OpenAI

In [2]:
# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a professional resume writer with 15 years of experience crafting
resumes that get interviews at top companies.

When given a person's details, produce a clean, modern, ATS-friendly resume in Markdown.
Follow this exact structure — no extra sections, no deviations:

# [Full Name]
**[Job Title]** | [email] | [location]

---

## Summary
[2–3 impactful sentences. Lead with their strongest qualifier. Include years of experience.
Quantify impact where possible. End with what they bring to their next role.]

## Skills
**[Category]:** [skill, skill, skill]  
**[Category]:** [skill, skill, skill]

## Experience

### [Job Title] — [Company] *([Duration])*
- [Strong action verb] + [what you did] + [measurable impact]
- [Strong action verb] + [what you did] + [measurable impact]
- [Strong action verb] + [what you did] + [measurable impact]

[Repeat for each role, most recent first]

## Education

### [Degree] — [Institution] *([Year])*

---

Write ONLY the resume. No preamble, no commentary, no code blocks."""

# ── One-shot example (multi-shot prompting) ────────────────────────────────────
SHOT_EXAMPLE = [
    {
        "role": "user",
        "content": (
            "Name: Priya Sharma\n"
            "Title: Senior Data Scientist\n"
            "Email: priya.sharma@email.com\n"
            "Location: Bangalore, India\n"
            "Skills: Python, SQL, R, PyTorch, Scikit-learn, Spark, AWS, Tableau\n"
            "Experience:\n"
            "- Senior Data Scientist at FinTech Corp (2021-Present): Led churn prediction model "
            "reducing churn by 18%. Built fraud detection pipeline processing 2M transactions/day.\n"
            "- Data Scientist at RetailAI (2018-2021): Built recommendation engine boosting "
            "average order value by 22%. Mentored 3 junior data scientists.\n"
            "Education: M.Sc. Statistics, IIT Delhi, 2018"
        )
    },
    {
        "role": "assistant",
        "content": (
            "# Priya Sharma\n"
            "**Senior Data Scientist** | priya.sharma@email.com | Bangalore, India\n\n"
            "---\n\n"
            "## Summary\n"
            "Results-driven Senior Data Scientist with 6+ years turning complex data into "
            "measurable business outcomes across fintech and retail. Proven track record in "
            "fraud detection, churn modelling, and recommendation systems — consistently "
            "delivering double-digit percentage improvements. Brings a rare combination of "
            "deep ML expertise and the communication skills to translate insights into product decisions.\n\n"
            "## Skills\n"
            "**Languages:** Python, SQL, R  \n"
            "**ML / DL:** PyTorch, Scikit-learn, XGBoost  \n"
            "**Data Engineering:** Apache Spark, AWS (S3, SageMaker, Redshift)  \n"
            "**Visualisation:** Tableau, Matplotlib, Seaborn\n\n"
            "## Experience\n\n"
            "### Senior Data Scientist — FinTech Corp *(2021–Present)*\n"
            "- Built churn prediction model (XGBoost + SHAP) that reduced customer churn by **18%**, saving $4.2M annually\n"
            "- Designed real-time fraud detection pipeline on Spark + AWS Kinesis processing **2M transactions/day** with <50ms latency\n"
            "- Partnered with product and engineering to ship 4 ML features to production in 12 months\n\n"
            "### Data Scientist — RetailAI *(2018–2021)*\n"
            "- Developed collaborative-filtering recommendation engine that increased average order value by **22%**\n"
            "- Reduced model training time by 60% by migrating batch jobs to distributed Spark on AWS EMR\n"
            "- Mentored 3 junior data scientists and established team's code review and MLOps best practices\n\n"
            "## Education\n\n"
            "### M.Sc. Statistics — IIT Delhi *(2018)*\n\n"
            "---"
        )
    }
]

print("Prompts loaded.")

Prompts loaded.


In [3]:
def build_messages(name, title, email, location, skills, experience, education):
    user_content = (
        f"Name: {name}\n"
        f"Title: {title}\n"
        f"Email: {email}\n"
        f"Location: {location}\n"
        f"Skills: {skills.strip()}\n"
        f"Experience:\n{experience.strip()}\n"
        f"Education:\n{education.strip()}"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        *SHOT_EXAMPLE,
        {"role": "user", "content": user_content},
    ]


def generate_resume(api_key, model_choice, name, title, email, location, skills, experience, education):
    if not name.strip() or not experience.strip():
        return "Please fill in at least your **Name** and **Work Experience**."

    if model_choice == "Llama 3.2 (local Ollama)":
        client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
        model  = "llama3.2"
    else:
        if not api_key.strip():
            return "Please enter your **OpenAI API key** to use GPT-4o-mini."
        client = OpenAI(api_key=api_key.strip())
        model  = "gpt-4o-mini"

    try:
        response = client.chat.completions.create(
            model=model,
            messages=build_messages(name, title, email, location, skills, experience, education),
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"


print("Functions ready.")

Functions ready.


In [4]:
DEMO = {
    "name":       "Alex Johnson",
    "title":      "Full Stack Developer",
    "email":      "alex.johnson@email.com",
    "location":   "New York, USA",
    "skills":     "Languages: Python, JavaScript, TypeScript\nFrameworks: React, FastAPI, Node.js\nCloud & Tools: AWS, Docker, PostgreSQL, Redis",
    "experience": (
        "- Senior Developer at XYZ Corp (2022–Present): \n"
        "  Built microservices platform now serving 500k daily users. Reduced API latency by 40%.\n"
        "  Led migration from monolith to Kubernetes, cutting infra costs by $120k/year.\n"
        "- Software Developer at ABC Startup (2020–2022): \n"
        "  Led full frontend rewrite in React, improving page load time by 60%.\n"
        "  Shipped 3 major product features end-to-end, each within 2-week sprint cycles."
    ),
    "education":  "B.Sc. Computer Science — State University, 2020",
}

with gr.Blocks(title="Resume / CV Generator", theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        "# Resume / CV Generator\n"
        "Enter your **OpenAI API key** and details, then click **Generate Resume**.  \n"
        "Demo data is pre-loaded — you can generate immediately or replace with your own."
    )

    with gr.Row():
        api_key = gr.Textbox(
            label="OpenAI API Key",
            type="password",
            placeholder="sk-proj-...  (leave blank if using Llama 3.2)",
            scale=3,
        )
        model = gr.Dropdown(
            choices=["GPT-4o-mini", "Llama 3.2 (local Ollama)"],
            value="GPT-4o-mini",
            label="Model",
            scale=1,
        )

    gr.Markdown("### Your Details")

    with gr.Row():
        name  = gr.Textbox(label="Full Name",  value=DEMO["name"],  placeholder="e.g. Alex Johnson")
        title = gr.Textbox(label="Job Title",  value=DEMO["title"], placeholder="e.g. Full Stack Developer")

    with gr.Row():
        email    = gr.Textbox(label="Email",    value=DEMO["email"],    placeholder="e.g. alex@email.com")
        location = gr.Textbox(label="Location", value=DEMO["location"], placeholder="e.g. New York, USA")

    skills = gr.Textbox(
        label="Skills",
        value=DEMO["skills"],
        lines=3,
        placeholder="Languages: Python, SQL\nFrameworks: React, FastAPI\nCloud: AWS, Docker",
    )
    experience = gr.Textbox(
        label="Work Experience",
        value=DEMO["experience"],
        lines=6,
        placeholder="- Senior Dev at XYZ Corp (2022–Present): Built X, achieved Y impact.\n- Dev at ABC (2020–2022): ...",
    )
    education = gr.Textbox(
        label="Education",
        value=DEMO["education"],
        lines=2,
        placeholder="B.Sc. Computer Science — State University, 2020",
    )

    btn = gr.Button("Generate Resume", variant="primary", size="lg")

    output = gr.Markdown(label="Your Resume")

    btn.click(
        fn=generate_resume,
        inputs=[api_key, model, name, title, email, location, skills, experience, education],
        outputs=output,
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
